# 实验一：静态页面与 API 数据采集

**平台：SmartPark DataLab**（https://115.159.159.82）

- Device：动态加载 → 找到 API → JSON → 分页采集全部设备
- Company：静态加载 → HTML → BeautifulSoup → “下一页” → 采集 3 页

## 0. 环境准备

目标站点使用自签名证书，因此 `requests` 需要 `verify=False`，并关闭对应的警告。

In [1]:
import json
import re
import warnings
from collections import Counter
from pathlib import Path
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

warnings.filterwarnings("ignore")

BASE_URL = "https://115.159.159.82"


# 任务一：采集 Device API

`/devices` 页面的设备数据**不在**初始 HTML 中（“查看网页源代码”搜不到设备名称），
通过 F12 → Network → Fetch/XHR 可以找到真正返回数据的请求：

- Request URL：`https://115.159.159.82/api/devices`
- Request Method：`GET`
- Query 参数：`page`、`page_size`

## 1.1 用 requests 复现第一页请求

In [2]:
api_url = BASE_URL + "/api/devices"

r = requests.get(
    api_url,
    params={"page": 1, "page_size": 50},
    timeout=10,
    verify=False,
)

print("status:", r.status_code)
print("url:", r.url)
print("content-type:", r.headers.get("Content-Type"))

status: 200
url: https://115.159.159.82/api/devices?page=1&page_size=50
content-type: application/json


In [3]:
data = r.json()

print(type(data))
print(data.keys())

<class 'dict'>
dict_keys(['page', 'page_size', 'total', 'items'])


In [4]:
print("当前页 page     =", data["page"])
print("每页条数 page_size =", data["page_size"])
print("设备总数 total     =", data["total"])
print("本页设备列表 items  共", len(data["items"]), "条")

print()
print(data["items"][0])

当前页 page     = 1
每页条数 page_size = 50
设备总数 total     = 5000
本页设备列表 items  共 50 条

{'device_id': 'D00000001', 'company_id': 'C000001', 'device_name': '循环泵-000001-001', 'device_type': 'pump', 'model': 'SP-PUM-800', 'zone': '智造一区', 'install_date': '2021-09-19', 'rated_power_kw': 71.2, 'sampling_interval_s': 30, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-01T03:58:00'}


## 1.2 计算总页数并采集全部设备

总页数由 API 返回的 `total` 和 `page_size` 计算，不写死。

In [5]:
total = data["total"]
page_size = data["page_size"]

total_pages = (total + page_size - 1) // page_size

print("total:", total)
print("page_size:", page_size)
print("total_pages:", total_pages)

total: 5000
page_size: 50
total_pages: 100


In [6]:
devices = []

for page in range(1, total_pages + 1):
    r = requests.get(
        api_url,
        params={"page": page, "page_size": page_size},
        timeout=10,
        verify=False,
    )
    page_data = r.json()
    devices.extend(page_data["items"])

print("device count:", len(devices))

device count: 5000


# 任务二：采集 Company 静态页面

`/companies` 页面在“查看网页源代码”中**能**搜到企业名称，属于静态加载，直接请求 HTML 解析。

## 2.1 请求第一页并定位企业记录

一条完整企业记录的最外层节点是 `<article class="company-card" data-company-id="C000001">`，
因此记录 selector 使用 `article.company-card`：class 语义明确、唯一对应企业卡片，且不依赖层级位置。

In [7]:
current_url = BASE_URL + "/companies"

r = requests.get(current_url, timeout=10, verify=False)
soup = BeautifulSoup(r.text, "html.parser")

cards = soup.select("article.company-card")
print("本页企业数：", len(cards))

本页企业数： 20


## 2.2 提取企业字段

- `company_id`：卡片节点的 `data-company-id` 属性；
- `name`：卡片内部 `h2 a` 的文本；
- `status`：卡片顶部 `.badge` 的文本；
- 行业 / 区域 / 设备数量：`dl > div` 下的 `dt/dd` 结构，先转字典再按键取值；
- `device_count` 去掉“台”字后转整数。

In [8]:
card = cards[0]

details = {}
for item in card.select("dl > div"):
    key = item.select_one("dt").get_text(strip=True)
    value = item.select_one("dd").get_text(strip=True)
    details[key] = value

print(details)

{'所属行业': '汽车零部件', '所在区域': '智造一区', '员工规模': '88 人', '设备数量': '6 台'}


In [9]:
def parse_company_card(card):
    details = {}
    for item in card.select("dl > div"):
        key = item.select_one("dt").get_text(strip=True)
        value = item.select_one("dd").get_text(strip=True)
        details[key] = value

    return {
        "company_id": card.get("data-company-id"),
        "name": card.select_one("h2 a").get_text(strip=True),
        "industry": details["所属行业"],
        "zone": details["所在区域"],
        "device_count": int(re.sub(r"[^0-9]", "", details["设备数量"])),
        "status": card.select_one(".company-card-top .badge").get_text(strip=True),
    }


companies_page1 = [parse_company_card(c) for c in cards]

print("本页企业数：", len(companies_page1))
print(companies_page1[0])

本页企业数： 20
{'company_id': 'C000001', 'name': '临港精工001有限公司', 'industry': '汽车零部件', 'zone': '智造一区', 'device_count': 6, 'status': '运营中'}


# 任务三：根据“下一页”连续采集 3 页 Company

页面底部的分页导航是 `<nav class="pagination">`，其中“下一页”是一个 `<a>` 标签，
`href` 形如 `/companies?page=2`（相对路径，需要 `urljoin` 补全）。
中间页“上一页”也是 `<a>`，因此用文本“下一页”进一步过滤，保证选中正确链接。

In [10]:
next_link = None
for a in soup.select("nav.pagination a"):
    if a.get_text(strip=True) == "下一页":
        next_link = a
        break

print(next_link.get_text(" ", strip=True))
print(next_link.get("href"))

next_url = urljoin(current_url, next_link.get("href"))
print(next_url)

下一页
/companies?page=2
https://115.159.159.82/companies?page=2


把「请求当前页 → 解析本页企业 → 找到下一页 href」组合进循环，连续采集 3 页。

In [11]:
companies = []
current_url = BASE_URL + "/companies"

for i in range(3):
    r = requests.get(current_url, timeout=10, verify=False)
    soup = BeautifulSoup(r.text, "html.parser")

    cards = soup.select("article.company-card")
    page_companies = [parse_company_card(c) for c in cards]
    companies.extend(page_companies)
    print(f"第{i + 1}页：{len(page_companies)}条")

    next_link = None
    for a in soup.select("nav.pagination a"):
        if a.get_text(strip=True) == "下一页":
            next_link = a
            break
    if next_link is None:
        break
    current_url = urljoin(current_url, next_link.get("href"))

print("Company 总数：", len(companies))

第1页：20条
第2页：20条
第3页：20条
Company 总数： 60


# 任务四：验证采集结果

## 4.1 数量与 ID 唯一性

In [12]:
print("companies:", len(companies))
print("unique company_id:", len(set(x["company_id"] for x in companies)))

print("devices:", len(devices))
print("unique device_id:", len(set(x["device_id"] for x in devices)))

companies: 60
unique company_id: 60
devices: 5000
unique device_id: 5000


## 4.2 Company–Device 一致性

用全部 Device 的 `company_id` 统计每家企业实际设备数，与 Company 页面中的 `device_count` 比较。

In [13]:
device_counts = Counter(x["company_id"] for x in devices)

mismatch_count = 0
for company in companies:
    cid = company["company_id"]
    expected = company["device_count"]
    actual = device_counts.get(cid, 0)
    if expected != actual:
        mismatch_count += 1
        print("不一致:", cid, "页面值", expected, "实际值", actual)

print("Company-Device 不一致企业数:", mismatch_count)

Company-Device 不一致企业数: 0


## 4.3 最终验证汇总

In [14]:
print("Company 总数:", len(companies))
print("唯一 company_id 数:", len(set(x["company_id"] for x in companies)))
print("Device 总数:", len(devices))
print("唯一 device_id 数:", len(set(x["device_id"] for x in devices)))
print("Company-Device 不一致企业数:", mismatch_count)

Company 总数: 60
唯一 company_id 数: 60
Device 总数: 5000
唯一 device_id 数: 5000
Company-Device 不一致企业数: 0


# 任务五：保存采集结果

In [15]:
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

with open(data_dir / "devices.json", "w", encoding="utf-8") as f:
    json.dump(devices, f, ensure_ascii=False, indent=2)

with open(data_dir / "companies.json", "w", encoding="utf-8") as f:
    json.dump(companies, f, ensure_ascii=False, indent=2)

print("数据已保存到 data/ 目录")

数据已保存到 data/ 目录
